# RQ2: 知识与数据的双重跨界（Paper-Level）

研究问题：知识跨界（topic distance）与数据跨界（dataset distance/atypicality）如何影响论文引用？

因变量：C3, C5
核心自变量：topic_distance, dataset_distance
控制变量：H_index, Academic_Age, Team_Size, Topic_Diversity

模型：OLS + NB（全作者 + 第一作者双栏）

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

from src.data_loader import load_config
from src.regression import run_ols, run_nb, save_results
from src.visualization import plot_forest_chart, save_fig
import pandas as pd

In [ ]:
config = load_config()
df_all = pd.read_csv(os.path.join('studies', 'RQ2', 'rq2_all_author_ready.csv'))
print(f'全作者数据维度: {df_all.shape}')

first_author_path = os.path.join('studies', 'RQ2', 'rq2_first_author_ready.csv')
df_first = pd.read_csv(first_author_path) if os.path.exists(first_author_path) else None
if df_first is not None:
    print(f'第一作者数据维度: {df_first.shape}')

## 1. 全作者 OLS 回归

In [ ]:
x_vars = ['topic_distance', 'dataset_distance']
controls = ['H_index', 'Academic_Age', 'Team_Size', 'Topic_Diversity']
controls = [c for c in controls if c in df_all.columns]

all_author_results = {}
for y_var in ['C3', 'C5']:
    for x_var in x_vars:
        if y_var in df_all.columns and x_var in df_all.columns:
            label = f'{y_var} ~ {x_var}'
            all_author_results[label] = run_ols(df_all, y_var, x_var, controls)
            if all_author_results[label]['converged']:
                print(f'{label}: coef={all_author_results[label]["coef"]:.4f}, p={all_author_results[label]["pvalue"]:.4f}')

## 2. 第一作者 OLS 回归

In [ ]:
if df_first is not None:
    first_author_results = {}
    for y_var in ['C3', 'C5']:
        for x_var in x_vars:
            if y_var in df_first.columns and x_var in df_first.columns:
                label = f'{y_var} ~ {x_var}'
                first_author_results[label] = run_ols(df_first, y_var, x_var, controls)
                if first_author_results[label]['converged']:
                    print(f'{label}: coef={first_author_results[label]["coef"]:.4f}, p={first_author_results[label]["pvalue"]:.4f}')

## 3. Forest Plot

In [ ]:
save_results(all_author_results, os.path.join('studies', 'RQ2', 'results', 'rq2_all_author_results.csv'))
if df_first is not None:
    save_results(first_author_results, os.path.join('studies', 'RQ2', 'results', 'rq2_first_author_results.csv'))
fig = plot_forest_chart(all_author_results, title='RQ2: Knowledge & Data Atypicality (All Authors)')
save_fig(fig, os.path.join('studies', 'RQ2', 'figures', 'rq2_forest_all.pdf'))
plt.show()